# Calibração de dificuldade: GPT-5.4 vs GPT-5.4 Mini

Teste curto para decidir se o deployment mini pode substituir o completo na avaliação de dificuldade. São avaliadas **3 questões aleatórias por tópico** (30 no conjunto atual), com a mesma rubrica e o mesmo prompt.

O GPT-5.4 completo é a referência operacional. Este notebook mede a concordância do mini com ele; não mede acurácia absoluta, que exigiria notas de dificuldade dadas por especialistas. Cada resultado é salvo imediatamente e a execução pode ser retomada.

**Regra sugerida para usar o mini:** concordância de classificação ≥ 85%, erro absoluto médio da nota ≤ 0,20 e concordância média das cinco dimensões ≥ 75%. Leia também os casos divergentes antes da decisão.

In [ ]:
import json
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


def find_root(start=Path.cwd()):
    for path in (start.resolve(), *start.resolve().parents):
        if (path / 'azure_openai_backend.py').exists():
            return path
    raise RuntimeError('Não encontrei a raiz do projeto.')


ROOT = find_root()
sys.path.insert(0, str(ROOT))
from azure_openai_backend import AzureOpenAIBackend

INPUT_PATH = ROOT / 'pipeline' / 'fase_3' / 'saida_fase3' / 'questoes_fase3.jsonl'
OUT_DIR = ROOT / 'pipeline' / 'fase_4' / 'calibracao_gpt54_vs_mini'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = OUT_DIR / 'checkpoint.jsonl'
CSV_PATH = OUT_DIR / 'comparacao.csv'

FULL_MODEL = 'gpt-5-4-petrobras'
MINI_MODEL = 'gpt-5-4-mini-petrobras'  # Altere apenas se o nome no gateway for outro.
SAMPLE_PER_TOPIC = 3
BATCH_SIZE = 5
SEED = 20260901
VERSION = 'difficulty-calibration-v1'

print(f'Entrada: {INPUT_PATH}')
print(f'Modelos: {FULL_MODEL} vs {MINI_MODEL}')

In [ ]:
DIMENSIONS = ('integracao', 'raciocinio', 'cognitivo', 'tecnico', 'distratores')
SYSTEM = '''Você avalia dificuldade intrínseca de questões técnicas de múltipla escolha para candidatos adequadamente preparados. Avalie o que é necessário para chegar ao gabarito editorial, não o tamanho do texto.

Pontue cada dimensão com inteiro de 1 a 3:
- integracao: 1=recordação direta; 2=aplicação de um conceito; 3=integra dois ou mais conceitos ou condições.
- raciocinio: 1=decisão direta; 2=comparação, causa/efeito ou duas etapas; 3=múltiplas inferências ou trade-offs indispensáveis.
- cognitivo: 1=lembrar/compreender; 2=aplicar; 3=analisar, avaliar ou decidir sob critérios concorrentes.
- tecnico: 1=conhecimento geral; 2=princípio técnico em contexto; 3=conhecimento especializado, condicionado ou quantitativo.
- distratores: 1=dois ou mais elimináveis cedo; 2=um eliminável cedo; 3=nenhum eliminável antes da resolução completa.
Retorne somente JSON válido.'''


def load_sample():
    questions = [json.loads(line) for line in INPUT_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    rng = random.Random(SEED)
    sample = []
    for topic in sorted({q['topico'] for q in questions}):
        pool = [q for q in questions if q['topico'] == topic]
        if len(pool) < SAMPLE_PER_TOPIC:
            raise ValueError(f'{topic}: menos de {SAMPLE_PER_TOPIC} questões.')
        sample.extend(rng.sample(pool, SAMPLE_PER_TOPIC))
    return sample


def payload(question):
    letters = 'ABCDEF'
    return {
        'id': question['id'], 'topico': question['topico'], 'enunciado': question['stem'],
        'alternativas': {letters[i]: text for i, text in enumerate(question['alternatives'])},
        'gabarito_editorial': letters[question['correct_answer_index']],
    }


def prompt(batch):
    schema = {'avaliacoes': [{'id': '...', 'integracao': 1, 'raciocinio': 1, 'cognitivo': 1, 'tecnico': 1, 'distratores': 1}]}
    return 'Avalie cada item independentemente.\nQUESTÕES:\n' + json.dumps([payload(q) for q in batch], ensure_ascii=False) + '\nJSON esperado:\n' + json.dumps(schema)


def label(score):
    return 'facil' if score <= 1.66 else ('media' if score <= 2.33 else 'dificil')


sample = load_sample()
print(f'Amostra: {len(sample)} questões, {SAMPLE_PER_TOPIC} por tópico.')
pd.DataFrame([{'id': q['id'], 'topico': q['topico'], 'difficulty_original': q.get('difficulty')} for q in sample])

In [ ]:
def completed():
    done = {}
    if CHECKPOINT.exists():
        for line in CHECKPOINT.read_text(encoding='utf-8').splitlines():
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if record.get('version') == VERSION and record.get('status') == 'ok':
                done[(record['model'], record['id'])] = record
    return done


def groups(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]


def validate(item, question_id):
    if not isinstance(item, dict) or item.get('id') != question_id:
        raise ValueError('ID ausente ou incorreto.')
    scores = {}
    for dimension in DIMENSIONS:
        value = item.get(dimension)
        if isinstance(value, bool) or not isinstance(value, int) or value not in (1, 2, 3):
            raise ValueError(f'{dimension} inválido.')
        scores[dimension] = value
    score = round(sum(scores.values()) / len(DIMENSIONS), 2)
    return scores | {'difficulty_score': score, 'difficulty_label': label(score)}


done = completed()
with CHECKPOINT.open('a', encoding='utf-8') as checkpoint:
    for model in (FULL_MODEL, MINI_MODEL):
        pending = [q for q in sample if (model, q['id']) not in done]
        print(f'{model}: {len(pending)} pendentes.')
        if not pending:
            continue
        backend = AzureOpenAIBackend(
            deployment=model, max_tokens=6000, reasoning_min_tokens=6000,
            reasoning_effort='low', request_timeout=180,
            cache_dir=str(OUT_DIR / 'cache'),
        )
        for batch in tqdm(list(groups(pending, BATCH_SIZE)), desc=model):
            now = datetime.now(timezone.utc).isoformat()
            try:
                response = backend.complete_json(prompt(batch), system=SYSTEM, max_tokens=6000)
                by_id = {x.get('id'): x for x in response.get('avaliacoes', []) if isinstance(x, dict)}
                batch_error = None
            except Exception as exc:
                by_id, batch_error = {}, f'{type(exc).__name__}: {exc}'
            for question in batch:
                base = {'id': question['id'], 'topico': question['topico'], 'model': model, 'version': VERSION, 'evaluated_at': now}
                try:
                    if batch_error:
                        raise RuntimeError(batch_error)
                    record = base | validate(by_id.get(question['id']), question['id']) | {'status': 'ok'}
                    done[(model, question['id'])] = record
                except Exception as exc:
                    record = base | {'status': 'erro', 'error': f'{type(exc).__name__}: {exc}'}
                checkpoint.write(json.dumps(record, ensure_ascii=False) + '\n')
                checkpoint.flush()

print(f'Checkpoint: {CHECKPOINT}')

In [ ]:
done = completed()
rows = []
for question in sample:
    full, mini = done.get((FULL_MODEL, question['id'])), done.get((MINI_MODEL, question['id']))
    if not (full and mini):
        continue
    row = {'id': question['id'], 'topico': question['topico'], 'score_gpt54': full['difficulty_score'], 'score_mini': mini['difficulty_score'], 'label_gpt54': full['difficulty_label'], 'label_mini': mini['difficulty_label']}
    row['mae_item'] = abs(row['score_gpt54'] - row['score_mini'])
    row['label_agree'] = row['label_gpt54'] == row['label_mini']
    for dimension in DIMENSIONS:
        row[f'{dimension}_agree'] = full[dimension] == mini[dimension]
    rows.append(row)

comparison = pd.DataFrame(rows)
if comparison.empty:
    print('Ainda não há pares concluídos. Execute a célula anterior.')
else:
    comparison.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
    label_agreement = comparison['label_agree'].mean()
    score_mae = comparison['mae_item'].mean()
    dimension_agreement = comparison[[f'{d}_agree' for d in DIMENSIONS]].mean().mean()
    approved = label_agreement >= .85 and score_mae <= .20 and dimension_agreement >= .75
    print(f'Pares concluídos: {len(comparison)}/{len(sample)}')
    print(f'Concordância da classificação: {label_agreement:.1%}')
    print(f'Erro absoluto médio da nota: {score_mae:.2f}')
    print(f'Concordância média das dimensões: {dimension_agreement:.1%}')
    print('DECISÃO SUGERIDA:', 'usar o mini nesta etapa' if approved else 'manter o modelo completo e revisar divergências')
    display(comparison[~comparison['label_agree']].sort_values('mae_item', ascending=False))
    print(f'CSV: {CSV_PATH}')